# Bronze Lookup DDL Ingestion

## Import Helper Functions

In [ ]:
from pathlib import Path

from src import *

# while it is allowed to use the import above, it is advised to list out what we have imported
# from src.config import *
# from src.spark_sql_magic import sql\
# from ingestion.metadata_utils import log_ingestion_metadata
# from src.raw_utils import checksum

## Import Libraries and Start Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window as W
import pyspark
import datetime
import json

spark = (
    SparkSession.builder
        .appName(JOB_NAMES["bronze_lookup_ddl_ingestion"])
        .getOrCreate()
)

## DDL

In [ ]:
for table_name, data in LOOKUP_TABLES.items():
    full_table = f"{CATALOG}.{BRONZE_NAMESPACE}.{table_name}"
    
    spark.sql(f"""
        CREATE TABLE IF NOT EXISTS {full_table} (
            {data['schema']}
        )
        USING ICEBERG
    """)
    print(f"{full_table} created.")

## Ingestion

In [ ]:
for table_name, data in LOOKUP_TABLES.items():
    full_table = f"{CATALOG}.{BRONZE_NAMESPACE}.{table_name}"
    
    df = spark.read.csv(data['file_path'], header=True)
    df.write.format("iceberg").mode("overwrite").saveAsTable(full_table)
    print(f"{table_name} ingested.")

    # metadata logging
    spark_path = full_table   # here it's table-based, not s3 object
    file_path = Path(data['file_path'])
    file_checksum = checksum(file_path)

    log_ingestion_metadata(
        spark=spark,
        table_name=f"{CATALOG}.{BRONZE_NAMESPACE}.{LOOKUP_METADATA_TABLE}",   # your metadata Iceberg table
        file_path=file_path,
        spark_path=spark_path,
        checksum=file_checksum
    ) 